In [17]:
import geopandas as gpd
import pandas as pd
import numpy as np

## Join Road Assets

In [20]:
potholes = gpd.read_file('datasets/potholes_with_target.gpkg')
road_assets = gpd.read_file('datasets/road_assets_cleaned.gpkg')

print(f"Potholes CRS: {potholes.crs}")
print(f"Road Assets CRS: {road_assets.crs}")

KeyboardInterrupt: 

In [4]:
# Join road_assets with potholes
potholes_joined = gpd.sjoin(potholes, road_assets, how='left', predicate='within')

print(f"Before join: {len(potholes)}")
print(f"After join: {len(potholes_joined)}")
print(f"Matched: {potholes_joined['ID_VOI_VOIRIE_AGR'].notna().sum()}")

Before join: 1027267
After join: 1030638
Matched: 627756


In [5]:
# Check for duplicates
print(f"Duplicate potholes: {len(potholes_joined) - len(potholes)}")

# Keep first match for duplicates
potholes_joined = potholes_joined.drop_duplicates(subset=['Date', 'Latitude', 'Longitude'], keep='first')
print(f"After removing duplicates: {len(potholes_joined)}")

Duplicate potholes: 3371
After removing duplicates: 910439


In [6]:
matched = potholes_joined['ID_VOI_VOIRIE_AGR'].notna().sum()
total = len(potholes_joined)
print(f"Matched: {matched} / {total} ({matched/total:.1%})")

Matched: 559375 / 910439 (61.4%)


In [7]:
# Split into matched and unmatched
matched_mask = potholes_joined['ID_VOI_VOIRIE_AGR'].notna()
potholes_matched = potholes_joined[matched_mask]
potholes_unmatched = potholes[~potholes.index.isin(potholes_matched.index)]

print(f"Matched: {len(potholes_matched)}")
print(f"Unmatched: {len(potholes_unmatched)}")

Matched: 559375
Unmatched: 467892


In [8]:
# Try nearest neighbour joining for unmatched
# First, get the unmatched potholes from original data
matched_indices = potholes_joined[potholes_joined['ID_VOI_VOIRIE_AGR'].notna()].index
potholes_unmatched = potholes[~potholes.index.isin(matched_indices)].copy()

print(f"Unmatched to process: {len(potholes_unmatched)}")

# Nearest join (requires both in projected CRS for accurate distances)
potholes_unmatched_proj = potholes_unmatched.to_crs("EPSG:32188")
road_assets_proj = road_assets.to_crs("EPSG:32188")

potholes_nearest = gpd.sjoin_nearest(
    potholes_unmatched_proj,
    road_assets_proj,
    how='left',
    distance_col='dist_to_road'
)

# Check distances
print(potholes_nearest['dist_to_road'].describe())

Unmatched to process: 467892
count    468366.000000
mean          3.298279
std          42.652698
min           0.000000
25%           0.467780
50%           1.890240
75%           4.222297
max       14151.189489
Name: dist_to_road, dtype: float64


In [9]:
MAX_DIST = 20  # meters

# Filter to reasonable distances
potholes_nearest_valid = potholes_nearest[potholes_nearest['dist_to_road'] <= MAX_DIST].copy()

print(f"Nearest matches within {MAX_DIST}m: {len(potholes_nearest_valid)}")
print(f"Dropped (too far): {len(potholes_nearest) - len(potholes_nearest_valid)}")

Nearest matches within 20m: 464686
Dropped (too far): 3680


In [12]:
# Convert nearest back to WGS84
potholes_nearest_valid = potholes_nearest_valid.to_crs("EPSG:4326")

# Get the matched ones from original join (remove duplicates first)
potholes_matched = potholes_joined[potholes_joined['ID_VOI_VOIRIE_AGR'].notna()].copy()
potholes_matched = potholes_matched.drop_duplicates(subset=['Date', 'Latitude', 'Longitude'], keep='first')

# Add dist_to_road column to matched (0 since they were within)
potholes_matched['dist_to_road'] = 0

# Combine
potholes_combined = pd.concat([potholes_matched, potholes_nearest_valid], ignore_index=True)

print(f"Total with road asset data: {len(potholes_combined)}")
print(f"Match rate: {len(potholes_combined) / len(potholes):.1%}")

Total with road asset data: 1024061
Match rate: 99.7%


In [13]:
# Clean up and save
print(potholes_combined.columns.tolist())

['Latitude', 'Longitude', 'source_year', 'Date', 'repeat', 'geometry', 'index_right', 'ID_VOI_VOIRIE_AGR', 'CATEGORIECHAUSSEE_REF', 'DATECONSTRUCTION', 'DATERESURFACAGE', 'MATERIAUCHAUSSEE_REF', 'TYPEFONDATION_REF', 'UTILISATION_REF', 'LAST_SURFACE_DATE', 'dist_to_road']


In [14]:
# Drop join artifacts
cols_to_drop = ['index_right']
potholes_combined = potholes_combined.drop(columns=[c for c in cols_to_drop if c in potholes_combined.columns])

# Check what we have
print(potholes_combined.info())

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1024061 entries, 0 to 1024060
Data columns (total 15 columns):
 #   Column                 Non-Null Count    Dtype         
---  ------                 --------------    -----         
 0   Latitude               1024061 non-null  float64       
 1   Longitude              1024061 non-null  float64       
 2   source_year            1024061 non-null  int64         
 3   Date                   1024061 non-null  datetime64[ms]
 4   repeat                 1024061 non-null  int64         
 5   geometry               1024061 non-null  geometry      
 6   ID_VOI_VOIRIE_AGR      1024061 non-null  float64       
 7   CATEGORIECHAUSSEE_REF  1024010 non-null  str           
 8   DATECONSTRUCTION       1024061 non-null  datetime64[ms]
 9   DATERESURFACAGE        223790 non-null   datetime64[ms]
 10  MATERIAUCHAUSSEE_REF   1024010 non-null  str           
 11  TYPEFONDATION_REF      0 non-null        object        
 12  UTILISATION_REF     

In [15]:
# Road age at time of repair
potholes_combined['road_age'] = (
    potholes_combined['Date'] - potholes_combined['DATECONSTRUCTION']
).dt.days / 365.25

# Years since last resurfacing (or construction if never resurfaced)
potholes_combined['years_since_surface'] = (
    potholes_combined['Date'] - potholes_combined['LAST_SURFACE_DATE']
).dt.days / 365.25

print(potholes_combined[['road_age', 'years_since_surface']].describe())

           road_age  years_since_surface
count  1.024061e+06         1.024061e+06
mean   5.331325e+01         4.216307e+01
std    2.799692e+01         3.319139e+01
min   -8.687201e+00        -8.780287e+00
25%    3.919233e+01         8.254620e+00
50%    5.518960e+01         4.925941e+01
75%    6.518549e+01         6.234634e+01
max    1.572512e+02         1.572512e+02


In [18]:
# Remove negative surface dates
# Set negative values to NaN (more honest than assuming 0)
potholes_combined.loc[potholes_combined['road_age'] < 0, 'road_age'] = np.nan
potholes_combined.loc[potholes_combined['years_since_surface'] < 0, 'years_since_surface'] = np.nan

print(f"Missing road_age: {potholes_combined['road_age'].isna().sum()}")
print(f"Missing years_since_surface: {potholes_combined['years_since_surface'].isna().sum()}")

Missing road_age: 72444
Missing years_since_surface: 197334


In [19]:
potholes_combined.to_file("datasets/potholes_with_road_assets.gpkg", driver="GPKG")
print(f"Saved {len(potholes_combined)} records")

Saved 1024061 records


## Join Road Condition

In [21]:
road_condition = pd.read_csv("datasets/road_condition_cleaned.csv")
# Check ID columns
print("Road condition ID sample:")
print(road_condition['ID_TRC'].head(10))

print("\nRoad assets ID sample:")
print(potholes_combined['ID_VOI_VOIRIE_AGR'].head(10))

Road condition ID sample:
0    1010191
1    1010192
2    1010194
3    1010195
4    1010197
5    1010201
6    1010202
7    1010203
8    1010204
9    1010205
Name: ID_TRC, dtype: int64

Road assets ID sample:
0    200304258.0
1    200304258.0
2    200080260.0
3    200304254.0
4    200080291.0
5    200308239.0
6    200099427.0
7    200099482.0
8    200128170.0
9    200148661.0
Name: ID_VOI_VOIRIE_AGR, dtype: float64


In [22]:
# Check for overlap
road_condition_ids = set(road_condition['ID_TRC'])
road_asset_ids = set(potholes_combined['ID_VOI_VOIRIE_AGR'].dropna())

overlap = road_condition_ids.intersection(road_asset_ids)
print(f"Road condition segments: {len(road_condition_ids)}")
print(f"Road asset segments in potholes: {len(road_asset_ids)}")
print(f"Overlapping IDs: {len(overlap)}")

Road condition segments: 35567
Road asset segments in potholes: 22634
Overlapping IDs: 0


In [23]:
# Road condition has 'Rue' column
print("Road condition street names:")
print(road_condition['Rue'].head(10))

# Check if road assets has street names
print("\nPotholes/road assets columns:")
print([c for c in potholes_combined.columns if 'rue' in c.lower() or 'street' in c.lower() or 'nom' in c.lower()])

Road condition street names:
0    Élie-Blanchard avenue
1    Élie-Blanchard avenue
2       Émile-Nelligan rue
3       Émile-Nelligan rue
4       Émile-Nelligan rue
5               Filion rue
6               Filion rue
7               Filion rue
8               Filion rue
9               Filion rue
Name: Rue, dtype: str

Potholes/road assets columns:
[]


In [27]:
# Need to use geobase since they dont share ids
geobase = gpd.read_file("datasets/road_network/geobase.geojson")

# Make sure CRS matches
geobase = geobase.to_crs(potholes_combined.crs)

# Geobase is lines, potholes are points - use nearest join
potholes_proj = potholes_combined.to_crs("EPSG:32188")
geobase_proj = geobase.to_crs("EPSG:32188")

# Join to nearest road segment
potholes_with_trc = gpd.sjoin_nearest(
    potholes_proj,
    geobase_proj[['ID_TRC', 'NOM_VOIE', 'geometry']],
    how='left',
    distance_col='dist_to_geobase'
)

print(f"Joined: {len(potholes_with_trc)}")
print(potholes_with_trc['dist_to_geobase'].describe())

Joined: 1024247
count    1.024247e+06
mean     4.523404e+00
std      3.696852e+00
min      8.605190e-06
25%      1.770716e+00
50%      3.704894e+00
75%      6.277868e+00
max      5.491568e+01
Name: dist_to_geobase, dtype: float64


In [28]:
# Check for duplicates from spatial join first
print(f"Before dedup: {len(potholes_with_trc)}")
potholes_with_trc = potholes_with_trc.drop_duplicates(subset=['Date', 'Latitude', 'Longitude'], keep='first')
print(f"After dedup: {len(potholes_with_trc)}")

# Load road condition
road_condition = pd.read_csv("datasets/road_condition_cleaned.csv")
road_condition['DateReleve'] = pd.to_datetime(road_condition['DateReleve'])

# Check overlap now
potholes_ids = set(potholes_with_trc['ID_TRC'].dropna())
condition_ids = set(road_condition['ID_TRC'])
print(f"Overlapping IDs: {len(potholes_ids.intersection(condition_ids))}")

Before dedup: 1024247
After dedup: 907266
Overlapping IDs: 19691


In [29]:
# Sort road condition by date
road_condition = road_condition.sort_values('DateReleve')

# For simplicity, let's first try: get the most recent assessment per segment
latest_condition = road_condition.groupby('ID_TRC').last().reset_index()

print(f"Segments with condition data: {len(latest_condition)}")

# Join to potholes
potholes_with_condition = potholes_with_trc.merge(
    latest_condition[['ID_TRC', 'Indice_PCI', 'Indice_IRI', 'DateReleve']],
    on='ID_TRC',
    how='left'
)

# Check match rate
matched = potholes_with_condition['Indice_PCI'].notna().sum()
print(f"Potholes with PCI data: {matched} / {len(potholes_with_condition)} ({matched/len(potholes_with_condition):.1%})")

Segments with condition data: 35567
Potholes with PCI data: 869791 / 907266 (95.9%)


In [30]:
# Rename the date column to avoid confusion
potholes_with_condition = potholes_with_condition.rename(columns={
    'DateReleve': 'ConditionAssessmentDate'
})

# Drop join artifacts
cols_to_drop = ['index_right', 'dist_to_geobase']
potholes_with_condition = potholes_with_condition.drop(
    columns=[c for c in cols_to_drop if c in potholes_with_condition.columns]
)

print(potholes_with_condition.info())

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 907266 entries, 0 to 907265
Data columns (total 22 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   Latitude                 907266 non-null  float64       
 1   Longitude                907266 non-null  float64       
 2   source_year              907266 non-null  int64         
 3   Date                     907266 non-null  datetime64[ms]
 4   repeat                   907266 non-null  int64         
 5   geometry                 907266 non-null  geometry      
 6   ID_VOI_VOIRIE_AGR        907266 non-null  float64       
 7   CATEGORIECHAUSSEE_REF    907217 non-null  str           
 8   DATECONSTRUCTION         907266 non-null  datetime64[ms]
 9   DATERESURFACAGE          196952 non-null  datetime64[ms]
 10  MATERIAUCHAUSSEE_REF     907217 non-null  str           
 11  TYPEFONDATION_REF        0 non-null       object        
 12  UTILISAT

In [31]:
# Convert back to WGS84 and save
potholes_with_condition = potholes_with_condition.to_crs("EPSG:4326")
potholes_with_condition.to_file("datasets/potholes_with_features.gpkg", driver="GPKG")

print(f"Saved {len(potholes_with_condition)} records")

Saved 907266 records


## Join with weather

In [32]:
weather = pd.read_csv("datasets/weather_cleaned.csv")
weather['Date'] = pd.to_datetime(weather['Date'])

# Sort by date
weather = weather.sort_values('Date').reset_index(drop=True)

print(weather.info())

<class 'pandas.DataFrame'>
RangeIndex: 3653 entries, 0 to 3652
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          3653 non-null   datetime64[us]
 1   MaxTemp       3604 non-null   float64       
 2   MinTemp       3613 non-null   float64       
 3   MeanTemp      3603 non-null   float64       
 4   Precip        3584 non-null   float64       
 5   SnowOnGround  1253 non-null   float64       
dtypes: datetime64[us](1), float64(5)
memory usage: 171.4 KB
None


In [33]:
# Freeze-thaw day = min below 0 AND max above 0
weather['freeze_thaw'] = ((weather['MinTemp'] < 0) & (weather['MaxTemp'] > 0)).astype(int)

print(f"Freeze-thaw days: {weather['freeze_thaw'].sum()} / {len(weather)}")

Freeze-thaw days: 742 / 3653


In [34]:
# Set date as index for rolling calculations
weather = weather.set_index('Date').sort_index()

# Rolling sums for different windows
weather['freeze_thaw_30d'] = weather['freeze_thaw'].rolling('30D').sum()
weather['freeze_thaw_60d'] = weather['freeze_thaw'].rolling('60D').sum()
weather['precip_30d'] = weather['Precip'].rolling('30D').sum()
weather['precip_60d'] = weather['Precip'].rolling('60D').sum()

# Reset index
weather = weather.reset_index()

print(weather[['Date', 'freeze_thaw_30d', 'freeze_thaw_60d', 'precip_30d', 'precip_60d']].head(60))

         Date  freeze_thaw_30d  freeze_thaw_60d  precip_30d  precip_60d
0  2016-01-01              1.0              1.0         0.2         0.2
1  2016-01-02              2.0              2.0         0.9         0.9
2  2016-01-03              3.0              3.0         5.0         5.0
3  2016-01-04              3.0              3.0         5.2         5.2
4  2016-01-05              3.0              3.0         5.2         5.2
5  2016-01-06              4.0              4.0         5.2         5.2
6  2016-01-07              5.0              5.0         5.2         5.2
7  2016-01-08              6.0              6.0         5.2         5.2
8  2016-01-09              6.0              6.0         8.1         8.1
9  2016-01-10              7.0              7.0        18.0        18.0
10 2016-01-11              7.0              7.0        18.0        18.0
11 2016-01-12              7.0              7.0        19.4        19.4
12 2016-01-13              7.0              7.0        19.6     

In [35]:
# Prepare for merge - match on date
potholes_with_condition['Date'] = pd.to_datetime(potholes_with_condition['Date'])

# Select weather columns to join
weather_features = weather[['Date', 'freeze_thaw_30d', 'freeze_thaw_60d', 'precip_30d', 'precip_60d', 'MeanTemp', 'SnowOnGround']]

# Merge
potholes_final = potholes_with_condition.merge(weather_features, on='Date', how='left')

# Check match rate
matched = potholes_final['freeze_thaw_30d'].notna().sum()
print(f"Potholes with weather data: {matched} / {len(potholes_final)} ({matched/len(potholes_final):.1%})")

Potholes with weather data: 907266 / 907266 (100.0%)


In [36]:
print(potholes_final.info())
print(potholes_final[['repeat', 'road_age', 'years_since_surface', 'Indice_PCI', 'Indice_IRI', 'freeze_thaw_30d', 'precip_30d']].describe())

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 907266 entries, 0 to 907265
Data columns (total 28 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   Latitude                 907266 non-null  float64       
 1   Longitude                907266 non-null  float64       
 2   source_year              907266 non-null  int64         
 3   Date                     907266 non-null  datetime64[ms]
 4   repeat                   907266 non-null  int64         
 5   geometry                 907266 non-null  geometry      
 6   ID_VOI_VOIRIE_AGR        907266 non-null  float64       
 7   CATEGORIECHAUSSEE_REF    907217 non-null  str           
 8   DATECONSTRUCTION         907266 non-null  datetime64[ms]
 9   DATERESURFACAGE          196952 non-null  datetime64[ms]
 10  MATERIAUCHAUSSEE_REF     907217 non-null  str           
 11  TYPEFONDATION_REF        0 non-null       object        
 12  UTILISAT

In [37]:
potholes_final.to_file("datasets/potholes_final.gpkg", driver="GPKG")
print(f"Saved {len(potholes_final)} records with all features")

Saved 907266 records with all features
